In [5]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from src.utils import setup_logging
from src.financial_statistics import FinancialStatistics
from src.portfolio import PortfolioEngine
from src.optimization import PortfolioOptimizer

In [6]:
# 1. Initialize Logging & Configuration
logger = setup_logging()
logger.info("Executing Phase 5: Portfolio Optimization Pipeline.")
rf_rate_assumption = 0.04

2026-07-12 23:30:46 [INFO] utils: Logging infrastructure successfully initialized.
2026-07-12 23:30:46 [INFO] 2781057939: Executing Phase 5: Portfolio Optimization Pipeline.


Writing logs to: /Users/admin/Desktop/Portofolio Analyzer/logs/portfolio_analyzer.log


In [8]:
# 2. Ingest Processed Simple Returns from Disk
simple_returns = pd.read_csv("data/processed/simple_returns.csv", index_col="Date", parse_dates=True)
asset_names = simple_returns.columns.tolist()

In [10]:
# 3. Generate Annualized Inputs via FinancialStatistics (Phase 5)
stats_engine = FinancialStatistics()
summary_stats = stats_engine.compute_summary_statistics(simple_returns, simple_returns) # Using simple_rets for both fields here for baseline returns mapping
matrices = stats_engine.get_matrices(simple_returns)

ann_returns = summary_stats["Annualized Return"]
ann_cov = matrices["covariance"]

2026-07-12 23:31:50 [INFO] financial_statistics: Generating baseline financial statistics profiles...
2026-07-12 23:31:50 [INFO] financial_statistics: Computing asset covariance and correlation matrices.


In [11]:
# 4. Initialize Optimization Engines
portfolio_engine = PortfolioEngine()
optimizer = PortfolioOptimizer(rf_rate=rf_rate_assumption)

In [12]:
# 5. Run Exact Solvers
max_sharpe_weights = optimizer.optimize_max_sharpe(ann_returns, ann_cov)
min_var_weights = optimizer.optimize_min_variance(ann_returns, ann_cov)

2026-07-12 23:32:06 [INFO] optimization: Executing Max Sharpe Optimization solver.
2026-07-12 23:32:06 [INFO] optimization: Executing Minimum Variance Optimization solver.


In [13]:
# 6. Run Random Portfolio Simulation for Visualizing Frontier (5000 paths)
sim_results = portfolio_engine.generate_random_portfolios(5000, ann_returns, ann_cov, rf_rate_assumption)

2026-07-12 23:32:20 [INFO] portfolio: Simulating 5000 random allocations for the Efficient Frontier.


In [14]:
# 7. Format and Print the Weights Table
weights_df = pd.DataFrame({
    "Asset": asset_names,
    "Max Sharpe Allocation": [f"{w*100:.2f}%" for w in max_sharpe_weights],
    "Min Variance Allocation": [f"{w*100:.2f}%" for w in min_var_weights]
})

print("\n=== OPTIMAL PORTFOLIO ALLOCATIONS ===")
display(weights_df)


=== OPTIMAL PORTFOLIO ALLOCATIONS ===


,Asset,Max Sharpe Allocation,Min Variance Allocation
0,AAPL,19.63%,2.81%
1,GLD,57.24%,3.09%
2,IEF,0.00%,88.75%
3,MSFT,23.13%,5.35%


In [15]:
# Compute exact coordinates for optimal portfolios
max_s_ret, max_s_vol, _ = portfolio_engine.calculate_metrics(max_sharpe_weights, ann_returns, ann_cov, rf_rate_assumption)
min_v_ret, min_v_vol, _ = portfolio_engine.calculate_metrics(min_var_weights, ann_returns, ann_cov, rf_rate_assumption)

# Trigger Visualizer
from src.visualization import FinancialVisualizer
visualizer = FinancialVisualizer()

visualizer.plot_efficient_frontier(
    sim_metrics=sim_results["metrics"],
    max_sharpe_point=(max_s_ret, max_s_vol),
    min_var_point=(min_v_ret, min_v_vol)
)
print("Efficient Frontier plot generated successfully and stored in the images/ directory.")

2026-07-12 23:33:47 [INFO] visualization: Generating Efficient Frontier scatter visualization.
2026-07-12 23:33:47 [INFO] visualization: Efficient Frontier plot exported successfully to: /Users/admin/Desktop/Portofolio Analyzer/images/efficient_frontier.png


Efficient Frontier plot generated successfully and stored in the images/ directory.
